[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/27.%20The%20Integrated%20Berth%20%26%20Crane%20Allocation%20Problem%20%28BAP-QCAP%29/P27-Tier-1_executed.ipynb)

---


# 27. The Integrated Berth and Crane Allocation Problem (BAP-QCAP)

## Tier 1: Mixed-Integer Programming Formulation

### Goal

Choose berth positions, service start times, and crane assignments for a set of vessels. The mixed-integer programming (MIP) model minimizes the total cost of vessel turnaround, where turnaround time runs from a vessel's arrival until its service is complete. Crane assignments affect service duration through a productivity-interference model.

### Inputs and outputs

The input is a planning instance containing vessel lengths, workloads, arrival times, hourly costs, crane productivities, quay length, an interference factor, and a planning horizon. The output is an optimal set of berth positions, service intervals, and crane assignments when the solver proves optimality within its time limit. A solver status other than `Optimal` is not reported as a proven optimum.

### Model assumptions and scope

- Vessel data and arrival times are known and deterministic.
- Berth allocation is static: each vessel is assigned a fixed berth interval, and that interval cannot be reused by another vessel during the planning horizon, even when their service periods do not overlap.
- Each crane is assigned to at most one vessel for the entire planning horizon. Crane assignments are static rather than time-dependent.
- Cranes have equal productivity in this example. The productivity of the $h$-th crane assigned to a vessel is reduced by the factor $1-\alpha h$, where $h=0$ for the first crane.
- The model does not represent crane travel, crane positions, vessel-specific crane reach, crane crossing, or time-dependent sharing of cranes. It also omits tides, berth-depth restrictions, setup times, and uncertainty.
- The maximum number of cranes per vessel is five, or the total number available if fewer than five cranes exist.

These assumptions define a small, deterministic planning model for studying the connection between berth allocation, crane counts, service duration, and turnaround cost. They are not a complete operational model of a container terminal.

### Required background and solution sequence

A decision variable represents a choice the optimizer is allowed to make. Continuous variables represent quantities such as position or time; binary variables represent yes-or-no choices. A constraint states which combinations of decisions are allowed. The objective function assigns a value to each allowed solution, and the optimizer seeks the allowed solution with the smallest objective value.

The model uses a one-hot selection: for each vessel, exactly one binary crane-count choice is 1, and all the other choices are 0. This makes it possible to represent a finite set of precomputed service durations with linear constraints. It also uses a big-$M$ construction for berth order. For each vessel pair, a binary variable activates one of the two possible left-to-right orders while relaxing the other inequality.

The notebook follows this sequence: define the data, construct the MIP, solve it with CBC through PuLP, extract the solution, visualize its components, and examine the effect of changing the interference factor. The mathematical model below makes the relationship between these steps explicit.

### Mathematical model

Let $V$ be the set of vessels and $K$ the set of cranes. For vessel $i\in V$, let $L_i$ be its length, $W_i$ its workload, $a_i$ its arrival time, and $c_i$ its cost per hour. Let $Q$ be the quay length, $H$ the planning horizon, $P$ the common crane productivity, and $\alpha$ the interference factor. Define $N=\{1,\ldots,\min(5,|K|)\}$ as the permitted crane counts.

For each vessel, $x_i$ is its berth position, $s_i$ and $f_i$ are its service start and finish times, and $y_{ik}$ is 1 if crane $k$ is assigned to vessel $i$. The binary variable $q_{in}$ is 1 if vessel $i$ receives exactly $n$ cranes. For each pair $i<j$, binary variable $o_{ij}$ selects which vessel is to the left of the other: $o_{ij}=1$ means vessel $i$ is to the left of vessel $j$. The position and time variables are continuous; the assignment, count-selection, and order variables are binary.

With $\tau_{in}$ denoting the service duration for vessel $i$ with $n$ cranes, the objective is

$$\min \sum_{i\in V} c_i(f_i-a_i).$$

The difference $f_i-a_i$ is turnaround time: it includes both waiting after arrival and service. Multiplying by $c_i$ converts that time into a vessel-specific turnaround cost.

The crane-count choice and its service duration are linked by

$$\sum_{n\in N}q_{in}=1,\qquad
\sum_{k\in K}y_{ik}=\sum_{n\in N}nq_{in},\qquad
f_i-s_i=\sum_{n\in N}\tau_{in}q_{in}\quad(i\in V),$$

where

$$\tau_{in}=\frac{W_i}{P\displaystyle\sum_{h=0}^{n-1}(1-\alpha h)}.$$

The sum in the denominator is the combined effective productivity of $n$ assigned cranes. The one-hot variables $q_{in}$ select exactly one permitted count, so the service-duration relationship is linear in the MIP even though the duration depends on a discrete crane-count choice. In the code, `calculate_service_time` computes each constant $\tau_{in}$ before the optimization model is solved.

The berth, resource, arrival, horizon, and variable-domain constraints are

$$0\le x_i\le Q-L_i,\qquad s_i\ge a_i,\qquad f_i\le H\quad(i\in V),$$

$$x_i+L_i\le x_j+Q(1-o_{ij}),\qquad
x_j+L_j\le x_i+Qo_{ij}\quad(i<j),$$

$$\sum_{i\in V}y_{ik}\le 1\quad(k\in K),\qquad
y_{ik}\in\{0,1\},\quad q_{in}\in\{0,1\},\quad o_{ij}\in\{0,1\}.$$

The berth inequalities enforce one of the two possible left-to-right orders for each vessel pair. The big-$M$ value is $Q$: when one inequality is relaxed, the other still enforces the selected order. Crane capacity is static over the planning horizon, as specified in the scope assumptions. The positive service duration implied by the selected crane count also ensures $f_i>s_i$. In the code, these quantities correspond to `positions`, `start_times`, `finish_times`, `crane_assignments`, `crane_count_choices`, and `spatial_order`.

The model has $O(|V||K|+|V||N|+|V|^2)$ binary variables, in addition to its continuous position and time variables. Since it contains binary decisions, the worst-case search effort of a general MIP solver can grow exponentially with the number of binary variables. This small example illustrates the formulation; it does not establish a universal vessel-count threshold for practical solvability.

### Example data

The instance contains three vessels: Vessel 1 has length 300 m, workload 800 TEU, and arrival time 08:00; Vessel 2 has length 300 m, workload 1,200 TEU, and arrival time 10:00; Vessel 3 has length 200 m, workload 400 TEU, and arrival time 12:00. There are 15 cranes, each with productivity 30 TEU per hour. The quay is 1,200 m long, the planning horizon is 48 hours, and the interference factor is $\alpha=0.1$.

### Interpreting results

A vessel's berth interval is from its reported position to that position plus its length. Its waiting time is its service start minus its arrival time; its service time is its finish minus its start; and its turnaround time is its finish minus its arrival. The objective is the sum of turnaround times weighted by each vessel's hourly cost. Different berth orders or crane assignments can have the same objective, so an optimal solution need not be unique.

In [ ]:
# Import the tools used to define, solve, and inspect the MIP.
import pulp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dataclasses import dataclass
from typing import List
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
# Dataclasses keep the instance data explicit and easy to inspect.
@dataclass
class Vessel:
    """A vessel requiring berth and crane services."""

    id: int
    length: float  # meters
    workload: float  # TEU
    arrival_time: float  # hours from the start of the planning day
    cost_per_hour: float = 1000.0

    def __post_init__(self):
        # This descriptive indicator is displayed with the instance data; it is not an optimization variable.
        time_window = max(1, 24 - self.arrival_time)
        self.priority = self.workload / (self.length * time_window)


@dataclass
class Crane:
    """A quay crane with a productivity rate."""

    id: int
    productivity: float  # TEU per hour
    position: float  # reference position along the quay, in meters


@dataclass
class BAPQCAPInstance:
    """All data required to define one BAP-QCAP planning instance."""

    vessels: List[Vessel]
    cranes: List[Crane]
    quay_length: float  # meters
    interference_factor: float
    planning_horizon: float  # hours

In [ ]:
# Create the numerical example specified in the problem description.
def create_concrete_example():
    """Create the instance with three vessels and fifteen cranes."""
    vessels = [
        Vessel(id=1, length=300, workload=800, arrival_time=8.0),
        Vessel(id=2, length=300, workload=1200, arrival_time=10.0),
        Vessel(id=3, length=200, workload=400, arrival_time=12.0),
    ]

    cranes = [
        Crane(id=i, productivity=30, position=i * 100)
        for i in range(15)
    ]

    return BAPQCAPInstance(
        vessels=vessels,
        cranes=cranes,
        quay_length=1200,
        interference_factor=0.1,
        planning_horizon=48,
    )


instance = create_concrete_example()

print("=== Problem instance ===")
print(f"Quay length: {instance.quay_length} m")
print(f"Available cranes: {len(instance.cranes)}")
print(f"Interference factor: {instance.interference_factor}")
print("\n=== Vessel details ===")
for vessel in instance.vessels:
    print(
        f"Vessel {vessel.id}: {vessel.length} m, {vessel.workload} TEU, "
        f"arrival {vessel.arrival_time:g}:00, priority indicator {vessel.priority:.2f}"
    )

In [ ]:
class BAPQCAPSolver:
    """Build and solve the mixed-integer BAP-QCAP formulation."""

    def __init__(self, instance: BAPQCAPInstance):
        self.instance = instance
        self.model = None
        self.solution = None
        self.positions = {}
        self.start_times = {}
        self.finish_times = {}
        self.crane_assignments = {}
        self.crane_count_choices = {}
        self.spatial_order = {}

    def calculate_service_time(self, vessel: Vessel, num_cranes: int) -> float:
        """Compute workload divided by interference-adjusted crane productivity."""
        if num_cranes < 1:
            raise ValueError("A vessel must be assigned at least one crane.")
        if not self.instance.cranes:
            raise ValueError("At least one crane is required.")

        productivity = self.instance.cranes[0].productivity
        if not all(
            np.isclose(crane.productivity, productivity)
            for crane in self.instance.cranes
        ):
            raise ValueError(
                "This formulation assumes that all cranes have equal productivity."
            )

        effective_productivity = productivity * sum(
            1 - self.instance.interference_factor * h
            for h in range(num_cranes)
        )
        if effective_productivity <= 0:
            raise ValueError(
                "The interference factor must leave positive productivity "
                "for every permitted crane count."
            )

        return vessel.workload / effective_productivity

    def build_model(self):
        """Create the objective, variables, and constraints of the MIP."""
        if not self.instance.vessels:
            raise ValueError("The instance must contain at least one vessel.")
        if not self.instance.cranes:
            raise ValueError("The instance must contain at least one crane.")
        if self.instance.quay_length <= 0 or self.instance.planning_horizon <= 0:
            raise ValueError("Quay length and planning horizon must be positive.")
        if any(vessel.length > self.instance.quay_length for vessel in self.instance.vessels):
            raise ValueError("Every vessel must fit within the quay length.")

        self.model = pulp.LpProblem("BAP-QCAP", pulp.LpMinimize)
        vessel_count = len(self.instance.vessels)
        crane_count = len(self.instance.cranes)
        permitted_counts = range(1, min(5, crane_count) + 1)

        for i, vessel in enumerate(self.instance.vessels):
            self.positions[i] = pulp.LpVariable(
                f"pos_{i}", 0, self.instance.quay_length - vessel.length
            )
            self.start_times[i] = pulp.LpVariable(
                f"start_{i}", vessel.arrival_time,
                self.instance.planning_horizon,
            )
            self.finish_times[i] = pulp.LpVariable(
                f"finish_{i}", vessel.arrival_time,
                self.instance.planning_horizon,
            )

            for k in range(crane_count):
                self.crane_assignments[(i, k)] = pulp.LpVariable(
                    f"crane_{i}_{k}", cat="Binary"
                )

            for n in permitted_counts:
                self.crane_count_choices[(i, n)] = pulp.LpVariable(
                    f"count_{i}_{n}", cat="Binary"
                )

        for i in range(vessel_count):
            for j in range(i + 1, vessel_count):
                self.spatial_order[(i, j)] = pulp.LpVariable(
                    f"left_{i}_{j}", cat="Binary"
                )

        # Turnaround includes both any waiting after arrival and the service itself.
        total_cost = pulp.lpSum(
            vessel.cost_per_hour
            * (self.finish_times[i] - vessel.arrival_time)
            for i, vessel in enumerate(self.instance.vessels)
        )
        self.model += total_cost, "Minimize_Total_Turnaround_Cost"

        for i, vessel in enumerate(self.instance.vessels):
            self.model += (
                pulp.lpSum(
                    self.crane_count_choices[(i, n)] for n in permitted_counts
                ) == 1,
                f"select_crane_count_{i}",
            )
            self.model += (
                pulp.lpSum(
                    self.crane_assignments[(i, k)] for k in range(crane_count)
                )
                == pulp.lpSum(
                    n * self.crane_count_choices[(i, n)] for n in permitted_counts
                ),
                f"link_crane_count_{i}",
            )
            self.model += (
                self.finish_times[i] - self.start_times[i]
                == pulp.lpSum(
                    self.calculate_service_time(vessel, n)
                    * self.crane_count_choices[(i, n)]
                    for n in permitted_counts
                ),
                f"service_duration_{i}",
            )
            self.model += (
                self.start_times[i] >= vessel.arrival_time,
                f"arrival_time_{i}",
            )

        # A pairwise binary choice enforces one left-to-right berth order.
        # Q is a sufficient big-M value because each vessel's berth interval
        # lies entirely within a quay of length Q.
        big_m = self.instance.quay_length
        for i, vessel_i in enumerate(self.instance.vessels):
            for j in range(i + 1, vessel_count):
                vessel_j = self.instance.vessels[j]
                order = self.spatial_order[(i, j)]
                self.model += (
                    self.positions[i] + vessel_i.length
                    <= self.positions[j] + big_m * (1 - order),
                    f"berth_order_{i}_{j}_first",
                )
                self.model += (
                    self.positions[j] + vessel_j.length
                    <= self.positions[i] + big_m * order,
                    f"berth_order_{i}_{j}_second",
                )

        # Crane assignments are static, so a crane cannot be assigned to
        # multiple vessels anywhere in this planning instance.
        for k in range(crane_count):
            self.model += (
                pulp.lpSum(
                    self.crane_assignments[(i, k)] for i in range(vessel_count)
                ) <= 1,
                f"crane_capacity_{k}",
            )

        return self.model

    def solve(self, time_limit=60):
        """Run CBC and record whether it proves the optimum."""
        if self.model is None:
            self.build_model()

        solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=time_limit)
        self.model.solve(solver)
        status = pulp.LpStatus[self.model.status]
        objective = pulp.value(self.model.objective)

        self.solution = {
            "status": status,
            "objective": objective if status == "Optimal" else None,
        }
        return self.solution

    def extract_solution(self):
        """Return vessel-level decisions only when optimality is proven."""
        if self.solution is None or self.solution["status"] != "Optimal":
            return None

        assignments = []
        for i, vessel in enumerate(self.instance.vessels):
            position = pulp.value(self.positions[i])
            start_time = pulp.value(self.start_times[i])
            finish_time = pulp.value(self.finish_times[i])
            assigned_cranes = [
                self.instance.cranes[k].id
                for k in range(len(self.instance.cranes))
                if pulp.value(self.crane_assignments[(i, k)]) > 0.5
            ]

            assignments.append(
                {
                    "vessel_id": vessel.id,
                    "position": position,
                    "start_time": start_time,
                    "finish_time": finish_time,
                    "crane_ids": assigned_cranes,
                    "num_cranes": len(assigned_cranes),
                    "service_time": finish_time - start_time,
                    "waiting_time": start_time - vessel.arrival_time,
                    "total_time": finish_time - vessel.arrival_time,
                }
            )

        return assignments


print("MIP solver class defined successfully.")

In [ ]:
# Build, solve, and extract the solution in the same order as the formulation.
def solve_bapqcap_instance(instance):
    """Solve one instance and display decisions when optimality is proven."""
    print("=== Solving BAP-QCAP with mixed-integer programming ===")
    solver = BAPQCAPSolver(instance)

    print("Building the MIP model...")
    model = solver.build_model()
    print(
        f"Model created with {len(model.variables())} variables and "
        f"{len(model.constraints)} constraints."
    )

    print("\nSolving the optimization problem...")
    solution = solver.solve(time_limit=30)
    print(f"\nSolver status: {solution['status']}")

    assignments = solver.extract_solution()
    if assignments is None:
        print("No proven optimal solution is available to report.")
        return solver, None

    print(f"Objective value: ${solution['objective']:.2f}")
    vessel_by_id = {vessel.id: vessel for vessel in instance.vessels}
    print("\n=== Optimal vessel assignments ===")
    for assignment in assignments:
        vessel = vessel_by_id[assignment["vessel_id"]]
        print(f"Vessel {assignment['vessel_id']}:")
        print(
            f"  Berth interval: {assignment['position']:.1f} m to "
            f"{assignment['position'] + vessel.length:.1f} m"
        )
        print(
            f"  Service: {assignment['start_time']:.1f} h to "
            f"{assignment['finish_time']:.1f} h"
        )
        print(
            f"  Crane IDs: {assignment['crane_ids']} "
            f"({assignment['num_cranes']} cranes)"
        )
        print(
            f"  Service time: {assignment['service_time']:.1f} h; "
            f"waiting time: {assignment['waiting_time']:.1f} h; "
            f"turnaround: {assignment['total_time']:.1f} h"
        )

    return solver, assignments


solver, assignments = solve_bapqcap_instance(instance)

In [ ]:
# The plots display spatial allocation, service intervals, and vessel-level metrics.
def visualize_solution(instance, assignments):
    """Plot the berth layout, schedule, metrics, and crane-count distribution."""
    if not assignments:
        print("No proven optimal solution is available to visualize.")
        return

    vessel_by_id = {vessel.id: vessel for vessel in instance.vessels}
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("BAP-QCAP Optimization Results", fontsize=16, fontweight="bold")
    colors = plt.cm.Set3(np.linspace(0, 1, len(assignments)))

    # Each berth rectangle shows the vessel's fixed spatial interval.
    ax1 = axes[0, 0]
    ax1.set_title("Berth Allocation", fontweight="bold")
    ax1.set_xlabel("Berth position (m)")
    ax1.set_ylabel("Vessel ID")
    for index, assignment in enumerate(assignments):
        vessel = vessel_by_id[assignment["vessel_id"]]
        rect = Rectangle(
            (assignment["position"], assignment["vessel_id"] - 0.4),
            vessel.length,
            0.8,
            facecolor=colors[index],
            edgecolor="black",
            linewidth=1.5,
        )
        ax1.add_patch(rect)
        ax1.text(
            assignment["position"] + vessel.length / 2,
            assignment["vessel_id"],
            f"V{assignment['vessel_id']}\n{assignment['num_cranes']} cranes",
            ha="center",
            va="center",
            fontweight="bold",
        )
    ax1.set_xlim(0, instance.quay_length)
    ax1.set_ylim(0, len(assignments) + 1)
    ax1.grid(True, alpha=0.3)

    # Bars begin at service start; dashed lines mark vessel arrivals.
    ax2 = axes[0, 1]
    ax2.set_title("Vessel Service Schedule", fontweight="bold")
    ax2.set_xlabel("Time (h from planning-day start)")
    ax2.set_ylabel("Vessel ID")
    for index, assignment in enumerate(assignments):
        vessel = vessel_by_id[assignment["vessel_id"]]
        ax2.barh(
            assignment["vessel_id"],
            assignment["service_time"],
            left=assignment["start_time"],
            height=0.6,
            color=colors[index],
            edgecolor="black",
        )
        ax2.axvline(
            x=vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.5,
            label="Arrival time" if index == 0 else None,
        )
    ax2.set_xlim(0, instance.planning_horizon)
    ax2.set_ylim(0, len(assignments) + 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Waiting, service, and turnaround are separate components of turnaround.
    ax3 = axes[1, 0]
    ax3.set_title("Vessel Time Components", fontweight="bold")
    vessel_ids = [a["vessel_id"] for a in assignments]
    waiting_times = [a["waiting_time"] for a in assignments]
    service_times = [a["service_time"] for a in assignments]
    total_times = [a["total_time"] for a in assignments]
    crane_counts = [a["num_cranes"] for a in assignments]
    x = np.arange(len(vessel_ids))
    width = 0.24
    ax3.bar(x - width, waiting_times, width, label="Waiting time (h)")
    ax3.bar(x, service_times, width, label="Service time (h)")
    ax3.bar(x + width, total_times, width, label="Turnaround time (h)")
    ax3.set_xlabel("Vessel ID")
    ax3.set_ylabel("Time (h)")
    ax3.set_xticks(x)
    ax3.set_xticklabels(vessel_ids)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # The pie chart summarizes how many vessels receive each crane count.
    ax4 = axes[1, 1]
    ax4.set_title("Crane Count by Vessel", fontweight="bold")
    crane_usage = {}
    for assignment in assignments:
        label = f"{assignment['num_cranes']} cranes"
        crane_usage[label] = crane_usage.get(label, 0) + 1
    ax4.pie(
        crane_usage.values(),
        labels=crane_usage.keys(),
        autopct="%1.1f%%",
        colors=plt.cm.Set3(np.linspace(0, 1, len(crane_usage))),
    )

    plt.tight_layout()
    plt.show()

    total_waiting = sum(waiting_times)
    total_service = sum(service_times)
    total_turnaround = sum(total_times)
    berth_length_used = sum(v.length for v in instance.vessels)
    print("\n=== Performance summary ===")
    print(f"Total waiting time: {total_waiting:.1f} h")
    print(f"Total service time: {total_service:.1f} h")
    print(f"Total turnaround time: {total_turnaround:.1f} h")
    print(f"Average turnaround per vessel: {np.mean(total_times):.1f} h")
    print(f"Total crane assignments: {sum(crane_counts)}")
    print(
        f"Berth-length utilization: "
        f"{berth_length_used / instance.quay_length * 100:.1f}%"
    )


visualize_solution(instance, assignments)

In [ ]:
# Vary only the interference factor to examine its effect under the same MIP.
def sensitivity_analysis(instance):
    """Resolve the instance for a specified set of interference factors."""
    print("=== Sensitivity analysis ===")
    interference_factors = [0.0, 0.05, 0.1, 0.15, 0.2]
    results = []

    for alpha in interference_factors:
        modified_instance = BAPQCAPInstance(
            vessels=instance.vessels,
            cranes=instance.cranes,
            quay_length=instance.quay_length,
            interference_factor=alpha,
            planning_horizon=instance.planning_horizon,
        )
        scenario_solver = BAPQCAPSolver(modified_instance)
        scenario_solver.build_model()
        solution = scenario_solver.solve(time_limit=10)
        scenario_assignments = scenario_solver.extract_solution()

        if scenario_assignments is None:
            results.append(
                {
                    "interference_factor": alpha,
                    "status": solution["status"],
                    "total_time": None,
                    "objective": None,
                }
            )
        else:
            total_time = sum(
                assignment["total_time"] for assignment in scenario_assignments
            )
            results.append(
                {
                    "interference_factor": alpha,
                    "status": solution["status"],
                    "total_time": total_time,
                    "objective": solution["objective"],
                }
            )

    valid_results = [result for result in results if result["objective"] is not None]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    if valid_results:
        alphas = [result["interference_factor"] for result in valid_results]
        times = [result["total_time"] for result in valid_results]
        costs = [result["objective"] for result in valid_results]

        ax1.plot(alphas, times, "bo-", linewidth=2, markersize=8)
        ax1.set_xlabel("Interference factor (α)")
        ax1.set_ylabel("Total turnaround time (h)")
        ax1.set_title("Turnaround time and crane interference")
        ax1.grid(True, alpha=0.3)

        ax2.plot(alphas, costs, "ro-", linewidth=2, markersize=8)
        ax2.set_xlabel("Interference factor (α)")
        ax2.set_ylabel("Total turnaround cost ($)")
        ax2.set_title("Cost and crane interference")
        ax2.grid(True, alpha=0.3)
    else:
        ax1.text(0.5, 0.5, "No proven optimal solutions", ha="center", va="center")
        ax2.text(0.5, 0.5, "No proven optimal solutions", ha="center", va="center")
    plt.tight_layout()
    plt.show()

    print("\nSensitivity analysis results:")
    print("Interference factor | Status | Total turnaround (h) | Objective ($)")
    print("-" * 72)
    for result in results:
        if result["objective"] is None:
            print(f"{result['interference_factor']:.2f} | {result['status']} | Not reported")
        else:
            print(
                f"{result['interference_factor']:.2f} | {result['status']} | "
                f"{result['total_time']:.2f} | ${result['objective']:.2f}"
            )

    return results


sensitivity_results = sensitivity_analysis(instance)

## Strengths, limitations, and use

The MIP expresses berth choices, crane counts, crane identities, service durations, and turnaround costs in one optimization model. For a feasible instance that CBC reports as `Optimal`, the solver has proven optimality for this formulation, subject to the solver's numerical tolerances. A time limit or a different solver status does not by itself establish optimality. More than one allocation may attain the same optimum.

The pairwise berth-order variables illustrate how a disjunction—vessel $i$ is left of vessel $j$ **or** vessel $j$ is left of vessel $i$—can be represented with binary variables and big-$M$ constraints. The crane-count selection variables make a finite set of nonlinear service-time choices usable in a linear MIP: each possible crane count has a precomputed duration, and exactly one count is selected.

The model's scope is intentionally limited. Static crane assignments prevent a crane from being reused by another vessel, even when their service intervals do not overlap. The model does not constrain crane positions or crossing, and it does not represent time-dependent berth sharing. These assumptions can make its allocations differ from those of a detailed terminal operation. The binary search space can also become difficult for larger instances; computational effort depends on the complete formulation, data, solver, and time limit rather than on a universal vessel-count cutoff.

This formulation is useful for learning how operational choices become MIP variables and constraints, and for solving small deterministic planning instances when a proven optimum is important. Results should be interpreted within the stated assumptions rather than as a complete operational plan.

In [ ]:
# Check that reported assignments satisfy the model's key relationships.
def final_summary(instance, solver, assignments):
    """Summarize the instance and validate an available optimal solution."""
    print("=== MIP formulation summary ===")
    print("\nProblem characteristics:")
    print(f"  Vessels: {len(instance.vessels)}")
    print(f"  Quay length: {instance.quay_length} m")
    print(f"  Available cranes: {len(instance.cranes)}")
    print(f"  Interference factor: {instance.interference_factor}")

    if not assignments:
        print("\nNo proven optimal solution is available for validation.")
        print(f"Solver status: {solver.solution['status']}")
        return

    vessel_by_id = {vessel.id: vessel for vessel in instance.vessels}
    total_cost = sum(
        assignment["total_time"]
        * vessel_by_id[assignment["vessel_id"]].cost_per_hour
        for assignment in assignments
    )
    total_workload = sum(vessel.workload for vessel in instance.vessels)
    total_cranes = sum(assignment["num_cranes"] for assignment in assignments)
    reported_objective = solver.solution["objective"]

    # These assertions check the extracted plan against the formulation.
    assert len(assignments) == len(instance.vessels)
    assert all(assignment["num_cranes"] >= 1 for assignment in assignments)
    assert all(
        assignment["num_cranes"] <= min(5, len(instance.cranes))
        for assignment in assignments
    )
    assert np.isclose(total_cost, reported_objective, rtol=1e-7, atol=1e-6)

    for assignment in assignments:
        vessel = vessel_by_id[assignment["vessel_id"]]
        assert assignment["start_time"] >= vessel.arrival_time - 1e-6
        assert assignment["finish_time"] <= instance.planning_horizon + 1e-6
        assert assignment["position"] >= -1e-6
        assert assignment["position"] + vessel.length <= instance.quay_length + 1e-6
        expected_service = solver.calculate_service_time(
            vessel, assignment["num_cranes"]
        )
        assert np.isclose(
            assignment["service_time"], expected_service, rtol=1e-7, atol=1e-6
        )

    for index, first in enumerate(assignments):
        first_vessel = vessel_by_id[first["vessel_id"]]
        for second in assignments[index + 1 :]:
            second_vessel = vessel_by_id[second["vessel_id"]]
            spatially_separated = (
                first["position"] + first_vessel.length
                <= second["position"] + 1e-6
                or second["position"] + second_vessel.length
                <= first["position"] + 1e-6
            )
            assert spatially_separated

    all_assigned_cranes = [
        crane_id
        for assignment in assignments
        for crane_id in assignment["crane_ids"]
    ]
    assert len(all_assigned_cranes) == len(set(all_assigned_cranes))

    print("\nOptimization results:")
    print(f"  Solver status: {solver.solution['status']}")
    print(f"  Total turnaround cost: ${total_cost:,.2f}")
    print(f"  Total workload: {total_workload} TEU")
    print(f"  Crane assignments: {total_cranes}")
    print(
        f"  Average vessel turnaround: "
        f"{np.mean([a['total_time'] for a in assignments]):.2f} h"
    )
    print(
        f"  Berth-length utilization: "
        f"{sum(v.length for v in instance.vessels) / instance.quay_length * 100:.1f}%"
    )
    print("\nModel interpretation:")
    print("  Berth-order binaries enforce spatial separation for every vessel pair.")
    print("  Crane-count choices determine interference-adjusted service durations.")
    print("  Turnaround cost includes waiting after arrival and service time.")
    print("  The validation checks the reported objective and key assignment constraints.")


final_summary(instance, solver, assignments)